Bermudan Swaption Pricing

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import sys
import json
import math
import numpy as np
from scipy.optimize import least_squares
from scipy.interpolate import CubicSpline
from scipy.stats import norm


class Curve:

    def __init__(self, zero_curve_data):
        mats = np.array([p["maturity"] for p in zero_curve_data], dtype=float)
        rates = np.array([p["rate"] for p in zero_curve_data], dtype=float)

        dfs = np.exp(-rates * mats)
        ldfs = np.log(dfs)

        t_k = np.concatenate([[0.0], mats])
        ldf_k = np.concatenate([[0.0], ldfs])

        self._cs = CubicSpline(t_k, ldf_k, bc_type="not-a-knot")
        self._t_max = float(mats[-1])
        self._ldf_max = float(self._cs(self._t_max))
        self._fwd_max = float(-self._cs(self._t_max, 1))

        self.mats = mats
        self.rates = rates
        self.dfs_knots = dfs
        self.fwds_knots = np.array([-float(self._cs(m, 1)) for m in mats])

    def log_df(self, t):
        t = float(t)

        if t <= 0.0:
            return 0.0

        if t <= self._t_max:
            return float(self._cs(t))

        return self._ldf_max - self._fwd_max * (t - self._t_max)

    def df(self, t):
        return math.exp(self.log_df(t))

    def fwd(self, t):
        t = float(t)

        if t < 1e-8:
            t = 1e-8

        if t <= self._t_max:
            return float(-self._cs(t, 1))

        return self._fwd_max

    def rate(self, t):
        if t <= 1e-10:
            return self.fwd(1e-8)

        return -self.log_df(t) / t


def hw_B(a, s, t):
    dt = t - s

    if dt <= 0.0:
        return 0.0

    if abs(a) < 1e-10:
        return dt

    return (1.0 - math.exp(-a * dt)) / a


def get_sigma_t(t, sigmas, bps):
    for k in range(len(sigmas)):
        if t < bps[k + 1]:
            return sigmas[k]

    return sigmas[-1]


def V_int(a, sigmas, bps, T0, T1, T2):
    if T1 <= T0 or abs(T2 - T1) < 1e-14:
        return 0.0

    if abs(a) < 1e-10:
        dt = T2 - T1
        total = 0.0

        for k in range(len(sigmas)):
            s0 = max(bps[k], T0)
            s1 = min(bps[k + 1], T1)

            if s1 <= s0:
                continue

            total += sigmas[k] ** 2 * dt ** 2 * (s1 - s0)

        return total

    fac = (math.exp(-a * T1) - math.exp(-a * T2)) ** 2 / a ** 2
    total = 0.0

    for k in range(len(sigmas)):
        s0 = max(bps[k], T0)
        s1 = min(bps[k + 1], T1)

        if s1 <= s0:
            continue

        total += sigmas[k] ** 2 * (
            math.exp(2 * a * s1) - math.exp(2 * a * s0)
        ) / (2 * a)

    return fac * total


def hw_lnA(a, sigmas, bps, t1, t2, curve):
    if t2 <= t1:
        return 0.0

    lnA = math.log(curve.df(t2) / curve.df(t1))
    lnA += hw_B(a, t1, t2) * curve.fwd(t1)
    lnA -= 0.5 * V_int(a, sigmas, bps, 0.0, t1, t2)

    return lnA


def hw_alpha(t, a, sigmas, bps, curve):
    f0t = curve.fwd(t)

    if t < 1e-10:
        return f0t

    phi = 0.0

    if abs(a) < 1e-10:
        for k in range(len(sigmas)):
            s0 = max(bps[k], 0.0)
            s1 = min(bps[k + 1], t)

            if s1 <= s0:
                continue

            phi += sigmas[k] ** 2 * (
                t * (s1 - s0) - 0.5 * (s1 ** 2 - s0 ** 2)
            )

    else:
        for k in range(len(sigmas)):
            s0 = max(bps[k], 0.0)
            s1 = min(bps[k + 1], t)

            if s1 <= s0:
                continue

            I1 = (
                math.exp(-a * (t - s1))
                - math.exp(-a * (t - s0))
            ) / a

            I2 = (
                math.exp(-2 * a * (t - s1))
                - math.exp(-2 * a * (t - s0))
            ) / (2 * a)

            phi += sigmas[k] ** 2 * (I1 - I2) / a

    return f0t + phi


def hw_zcbo_put(a, sigmas, bps, T_opt, T_bond, K_bond, curve):
    sp2 = V_int(a, sigmas, bps, 0.0, T_opt, T_bond)
    sp = math.sqrt(max(sp2, 0.0))

    P0T = curve.df(T_opt)
    P0S = curve.df(T_bond)

    if sp < 1e-14:
        return max(K_bond * P0T - P0S, 0.0)

    h = math.log(P0S / (P0T * K_bond)) / sp + sp / 2.0

    return (
        K_bond * P0T * norm.cdf(-h + sp)
        - P0S * norm.cdf(-h)
    )


def hw_european_swaption(
    a,
    sigmas,
    bps,
    T_exp,
    sw_end,
    fixed_rate,
    notional,
    curve
):
    n = int(round(sw_end - T_exp))

    if n <= 0:
        return 0.0

    pmts = [T_exp + i for i in range(1, n + 1)]

    c = [fixed_rate] * n
    c[-1] += 1.0

    lnAs = [
        hw_lnA(a, sigmas, bps, T_exp, ti, curve)
        for ti in pmts
    ]

    Bs = [hw_B(a, T_exp, ti) for ti in pmts]

    def bval(rt):
        return sum(
            c[i] * math.exp(lnAs[i] - Bs[i] * rt)
            for i in range(n)
        )

    r_lo, r_hi = -0.5, 1.5

    if (bval(r_lo) - 1.0) * (bval(r_hi) - 1.0) > 0.0:
        return 0.0

    for _ in range(100):
        rm = 0.5 * (r_lo + r_hi)
        fm = bval(rm) - 1.0

        if abs(fm) < 1e-13:
            break

        if (bval(r_lo) - 1.0) * fm < 0:
            r_hi = rm
        else:
            r_lo = rm

    r_star = 0.5 * (r_lo + r_hi)

    total = sum(
        c[i] * hw_zcbo_put(
            a,
            sigmas,
            bps,
            T_exp,
            pmts[i],
            math.exp(lnAs[i] - Bs[i] * r_star),
            curve
        )
        for i in range(n)
    )

    return notional * total


def hw_normal_vol(a, sigmas, bps, T_exp, tenor, curve):
    sw_end = T_exp + tenor

    n = int(round(tenor))
    pmts = [T_exp + i for i in range(1, n + 1)]

    P_T = curve.df(T_exp)
    P_S = curve.df(sw_end)

    ann = sum(curve.df(ti) for ti in pmts)

    if ann < 1e-14 or T_exp < 1e-10:
        return 0.0

    fwd = (P_T - P_S) / ann

    price = hw_european_swaption(
        a,
        sigmas,
        bps,
        T_exp,
        sw_end,
        fwd,
        1.0,
        curve
    )

    return price / (ann * math.sqrt(T_exp / (2.0 * math.pi)))


def calibrate_hw(
    swaption_vols,
    curve,
    bps,
    n_sigmas,
    x0_warm=None,
    max_nfev=2000,
    n_starts=None
):
    market_vols = np.array(
        [v["vol_bps"] for v in swaption_vols],
        dtype=float
    )

    expiries = [v["expiry"] for v in swaption_vols]
    tenors = [v["tenor"] for v in swaption_vols]

    def residuals(params):
        a = params[0]
        sigmas = list(params[1:])

        if a < 1e-6 or any(s < 1e-9 for s in sigmas):
            return np.full(len(market_vols), 1e4)

        res = []

        for exp, ten in zip(expiries, tenors):
            try:
                res.append(
                    hw_normal_vol(
                        a,
                        sigmas,
                        bps,
                        exp,
                        ten,
                        curve
                    ) * 1e4
                )

            except Exception:
                res.append(0.0)

        return np.array(res) - market_vols

    lb = [1e-6] + [1e-9] * n_sigmas
    ub = [2.0] + [0.20] * n_sigmas

    avg_vol = float(np.mean(market_vols)) * 1e-4

    all_starts = []

    if x0_warm is not None:
        all_starts.append(np.clip(x0_warm, lb, ub))

    all_starts += [
        np.clip([0.03] + [avg_vol] * n_sigmas, lb, ub),
        np.clip([0.05] + [0.008] * n_sigmas, lb, ub),
        np.clip([0.01] + [avg_vol] * n_sigmas, lb, ub),
        np.clip([0.10] + [0.007] * n_sigmas, lb, ub),
    ]

    starts = (
        all_starts
        if n_starts is None
        else all_starts[:n_starts]
    )

    best_cost = np.inf
    best_x = None

    for x0 in starts:
        try:
            res = least_squares(
                residuals,
                x0,
                bounds=(lb, ub),
                method="trf",
                ftol=1e-8,
                xtol=1e-8,
                gtol=1e-8,
                max_nfev=max_nfev,
            )

            if res.cost < best_cost:
                best_cost = res.cost
                best_x = res.x.copy()

        except Exception:
            pass

        if best_cost < 1e-6:
            break

    if best_x is None:
        best_x = np.array(starts[-1])

    return float(best_x[0]), list(best_x[1:])


def price_bermudan_tree(
    a,
    sigmas,
    bps,
    notional,
    strike,
    exercise_dates,
    swap_end,
    curve,
    n_steps_per_year=50
):
    ex_dates = sorted(float(e) for e in exercise_dates)

    dt_base = 1.0 / n_steps_per_year

    req = sorted({0.0} | set(ex_dates))

    tg = []

    for i in range(len(req) - 1):
        t0, t1 = req[i], req[i + 1]

        n_sub = max(
            1,
            int(round((t1 - t0) / dt_base))
        )

        pts = np.linspace(t0, t1, n_sub + 1)

        tg.extend(
            pts[1:].tolist()
            if i > 0
            else pts.tolist()
        )

    tg = sorted({round(t, 10) for t in tg})

    avg_sig = float(np.mean(sigmas))

    dx = max(
        avg_sig * math.sqrt(3.0 * dt_base),
        1e-4
    )

    if abs(a) >= 1e-8:
        j_max = max(
            15,
            min(
                200,
                int(math.ceil(0.1835 / (a * dt_base)))
            )
        )
    else:
        j_max = 60

    n_nodes = 2 * j_max + 1

    x_arr = (
        np.arange(-j_max, j_max + 1, dtype=float)
        * dx
    )

    alpha_arr = np.array([
        hw_alpha(t, a, sigmas, bps, curve)
        for t in tg
    ])

    ex_cache = {}

    for t_ex in ex_dates:
        n_p = int(round(swap_end - t_ex))

        pmts = [
            t_ex + k
            for k in range(1, n_p + 1)
        ]

        lnAs = np.array([
            hw_lnA(a, sigmas, bps, t_ex, ti, curve)
            for ti in pmts
        ])

        Bvals = np.array([
            hw_B(a, t_ex, ti)
            for ti in pmts
        ])

        lnA_e = hw_lnA(
            a,
            sigmas,
            bps,
            t_ex,
            swap_end,
            curve
        )

        B_e = hw_B(a, t_ex, swap_end)

        ex_cache[t_ex] = (
            lnAs,
            Bvals,
            lnA_e,
            B_e,
            n_p
        )

    def swap_val_vec(t_ex, r_arr):
        if t_ex not in ex_cache:
            return np.zeros(len(r_arr))

        lnAs, Bvals, lnA_e, B_e, n_p = ex_cache[t_ex]

        if n_p <= 0:
            return np.zeros(len(r_arr))

        val = 1.0 - np.exp(lnA_e - B_e * r_arr)

        val -= strike * np.sum(
            np.exp(
                lnAs[:, None]
                - Bvals[:, None] * r_arr
            ),
            axis=0
        )

        return notional * val

    ex_idx_map = {}

    for t_ex in ex_dates:
        for i, t in enumerate(tg):
            if abs(t - t_ex) < 1e-9 and t > 1e-10:
                ex_idx_map[i] = t_ex
                break

    last_ex = ex_dates[-1]

    last_idx = next(
        i for i, t in enumerate(tg)
        if abs(t - last_ex) < 1e-9
    )

    r_last = x_arr + alpha_arr[last_idx]

    V = np.maximum(
        swap_val_vec(last_ex, r_last),
        0.0
    )

    for step in range(last_idx - 1, -1, -1):
        t_c = tg[step]
        t_n = tg[step + 1]

        dt = t_n - t_c

        r_c = x_arr + alpha_arr[step]

        sig_c = get_sigma_t(
            t_c,
            sigmas,
            bps
        )

        if abs(a) < 1e-10:
            decay = 1.0
            var_x = sig_c ** 2 * dt

        else:
            decay = math.exp(-a * dt)

            var_x = (
                sig_c ** 2 / (2 * a)
            ) * (
                1.0 - math.exp(-2 * a * dt)
            )

        mu = x_arr * decay

        k = np.round(mu / dx).astype(int)

        k = np.clip(
            k,
            -j_max + 1,
            j_max - 1
        )

        k_m = k.copy()
        k_u = k + 1
        k_d = k - 1

        top = k >= j_max - 1

        k_u[top] = j_max
        k_m[top] = j_max - 1
        k_d[top] = j_max - 2

        bot = k <= -j_max + 1

        k_u[bot] = -j_max + 2
        k_m[bot] = -j_max + 1
        k_d[bot] = -j_max

        x_m = k_m.astype(float) * dx

        eta = mu - x_m
        edx = eta / dx

        spd = (var_x + eta ** 2) / dx ** 2

        p_u = 0.5 * (spd + edx)
        p_d = 0.5 * (spd - edx)
        p_m = 1.0 - p_u - p_d

        bad = (
            (p_u < 0)
            | (p_d < 0)
            | (p_m < 0)
        )

        if np.any(bad):
            pu2 = np.maximum(
                0.0,
                1 / 6 + 0.5 * edx ** 2 + 0.5 * edx
            )

            pd2 = np.maximum(
                0.0,
                1 / 6 + 0.5 * edx ** 2 - 0.5 * edx
            )

            pm2 = np.maximum(
                0.0,
                2 / 3 - edx ** 2
            )

            s = np.where(
                pu2 + pm2 + pd2 > 1e-12,
                pu2 + pm2 + pd2,
                1.0
            )

            p_u = np.where(bad, pu2 / s, p_u)
            p_d = np.where(bad, pd2 / s, p_d)
            p_m = np.where(bad, pm2 / s, p_m)

        iu = np.clip(
            k_u + j_max,
            0,
            n_nodes - 1
        )

        im = np.clip(
            k_m + j_max,
            0,
            n_nodes - 1
        )

        id_ = np.clip(
            k_d + j_max,
            0,
            n_nodes - 1
        )

        disc = np.exp(-r_c * dt)

        V_new = disc * (
            p_u * V[iu]
            + p_m * V[im]
            + p_d * V[id_]
        )

        if step in ex_idx_map:
            V_new = np.maximum(
                V_new,
                swap_val_vec(ex_idx_map[step], r_c)
            )

        V = V_new

    return float(V[j_max])


def main():
    data = json.loads(sys.stdin.read())

    zero_curve_data = data["zero_curve"]
    swaption_vols_data = data["swaption_vols"]
    spec = data["bermudan_spec"]

    notional = spec["notional"]
    strike = spec["strike"]

    exercise_dates = sorted(spec["exercise_dates"])
    swap_end = spec["swap_end"]

    bps = [0.0] + [
        float(e)
        for e in exercise_dates
    ]

    if bps[-1] < swap_end:
        bps.append(float(swap_end))

    n_sigmas = len(bps) - 1

    N_STEPS = 50

    curve = Curve(zero_curve_data)

    curve_output = [
        {
            "maturity": float(curve.mats[i]),
            "discount_factor": round(
                float(curve.dfs_knots[i]),
                4
            ),
            "forward_rate": round(
                float(curve.fwds_knots[i]),
                4
            ),
        }
        for i in range(len(curve.mats))
    ]

    a_cal, sig_cal = calibrate_hw(
        swaption_vols_data,
        curve,
        bps,
        n_sigmas,
        max_nfev=2000
    )

    x_base = [a_cal] + sig_cal

    calib_params = {
        "mean_reversion": round(float(a_cal), 6)
    }

    for i, s in enumerate(sig_cal):
        calib_params[f"sigma_{i + 1}"] = round(float(s), 6)

    calibration = {
        "model": "Hull-White",
        "parameters": calib_params
    }

    base_price = price_bermudan_tree(
        a_cal,
        sig_cal,
        bps,
        notional,
        strike,
        exercise_dates,
        swap_end,
        curve,
        N_STEPS
    )

    vega_results = []

    for vol_entry in swaption_vols_data:
        bumped = [
            {
                "expiry": v["expiry"],
                "tenor": v["tenor"],
                "vol_bps": v["vol_bps"] + (
                    1
                    if (
                        v["expiry"] == vol_entry["expiry"]
                        and v["tenor"] == vol_entry["tenor"]
                    )
                    else 0
                ),
            }
            for v in swaption_vols_data
        ]

        a_b, sig_b = calibrate_hw(
            bumped,
            curve,
            bps,
            n_sigmas,
            x0_warm=x_base,
            max_nfev=100,
            n_starts=1
        )

        bp = price_bermudan_tree(
            a_b,
            sig_b,
            bps,
            notional,
            strike,
            exercise_dates,
            swap_end,
            curve,
            N_STEPS
        )

        vega_results.append({
            "expiry": vol_entry["expiry"],
            "tenor": vol_entry["tenor"],
            "vega_dollars_per_bp": round(
                float(bp - base_price),
                4
            ),
        })

    delta_results = []

    for i_mat in range(len(zero_curve_data)):
        bumped_zc = [
            {
                "maturity": zc["maturity"],
                "rate": zc["rate"] + (
                    0.0001
                    if j == i_mat
                    else 0.0
                )
            }
            for j, zc in enumerate(zero_curve_data)
        ]

        b_curve = Curve(bumped_zc)

        dp = price_bermudan_tree(
            a_cal,
            sig_cal,
            bps,
            notional,
            strike,
            exercise_dates,
            swap_end,
            b_curve,
            N_STEPS
        )

        delta_results.append({
            "maturity": zero_curve_data[i_mat]["maturity"],
            "delta_dollars_per_bp": round(
                float(dp - base_price),
                4
            ),
        })

    output = {
        "curve": curve_output,
        "calibration": calibration,
        "price_dollars": round(float(base_price), 2),
        "vega": vega_results,
        "delta": delta_results,
    }

    print(json.dumps(output, indent=2))


if __name__ == "__main__":
    main()